# Adaptive retraining strategy experiment for Google Colab

This notebook runs a reproducible offline pilot for the retraining protocol discussed in the project notes. It uses real alanine-dipeptide backbone dihedral trajectories from `mdshare` as the equilibrium reference and as a replay stream for debugging trigger, training-window, reweighting, and training-budget decisions.

Scope note: `mdshare` trajectories are unbiased static trajectories, so this notebook cannot replace a coupled OpenMM/PLUMED adaptive metadynamics run. It is intentionally structured so that the `ReplayStream` source can later be replaced by a real biased MD stream that reports `(phi, psi, V_bias, wall_time)` per frame. The metrics, grid, and budget accounting remain the same.

## 1. Colab setup

Run this cell in Colab from the repository root. If the notebook is opened outside a clone, it clones the public repository first. The project is installed through Poetry, then `mdshare` is installed into the same environment.

In [ ]:
from pathlib import Path
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB and not Path("pyproject.toml").exists():
    !git clone https://github.com/Komputerowe-Projektowanie-Lekow/pmarlo.git
    %cd pmarlo

if IN_COLAB:
    %pip install -q poetry
    !poetry config virtualenvs.create false
    !poetry install --extras "analysis mlcv" --with tests --no-interaction
    !poetry run pip install mdshare
else:
    print("Local run detected. Use: poetry run jupyter notebook example_programs/13_adaptive_retraining_colab.ipynb")

## 2. Imports and experiment constants

The default configuration executes the full 3 x 4 x 3 factorial grid with 10 replicas. Reduce `REPLICATES_PER_CONDITION` or `BUDGET_FRAMES` only for a quick Colab smoke test.

In [ ]:
from __future__ import annotations

import itertools
import json
import math
import time
from collections import deque
from dataclasses import asdict, dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import mdshare
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "example_programs" / "programs_outputs" / "adaptive_retraining_colab"
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RNG_SEED = 20260517
BINS = 48
TEMPERATURE_K = 300.0
LAG = 10
VAMP_COMPONENTS = 2

BUDGET_FRAMES = 25_000
REPLICATES_PER_CONDITION = 10
MONITOR_WINDOW = 2_000
MONITOR_STRIDE = 500
TRAIN_WINDOW = 8_000
FIXED_TRIGGER_PERIOD = 5_000
THRESHOLD_DELTA = 0.08

ADWIN_DELTA = 0.01
ADWIN_MIN_WINDOW = 12
ADWIN_MAX_WINDOW = 80

HILL_STRIDE = 250
HILL_HEIGHT_KBT = 0.04
HILL_SIGMA_RAD = 0.25
BETA = 1.0

MAX_PAIRS_PER_EPOCH = 2_048
EARLY_STOPPING_PATIENCE = 8
EARLY_STOPPING_EPS = 1.0e-4

TRAINING_POLICIES = {
    "Fixed-50ep": {"max_epochs": 50, "early_stopping": False},
    "Fixed-200ep": {"max_epochs": 200, "early_stopping": False},
    "EarlyStopping": {"max_epochs": 200, "early_stopping": True},
}

TRIGGER_POLICIES = ["Fixed-T", "Threshold-delta", "ADWIN"]
DATA_POLICIES = ["Full", "Window-W", "Reweighted-Full", "Reweighted-Window"]

print(f"Output directory: {OUTPUT_DIR}")

## 3. Load mdshare alanine-dipeptide data

`mdshare.fetch()` downloads `alanine-dipeptide-3x250ns-backbone-dihedrals.npz`, which contains three real equilibrium trajectories with shape `(250000, 2)`. Angles are wrapped to `[-pi, pi)`.

In [ ]:
def wrap_periodic(angles: np.ndarray) -> np.ndarray:
    wrapped = (np.asarray(angles, dtype=np.float64) + np.pi) % (2.0 * np.pi) - np.pi
    return wrapped.astype(np.float64, copy=False)


def load_mdshare_dihedrals(data_dir: Path) -> list[np.ndarray]:
    filename = mdshare.fetch(
        "alanine-dipeptide-3x250ns-backbone-dihedrals.npz",
        working_directory=str(data_dir),
    )
    with np.load(filename) as handle:
        trajectories = [wrap_periodic(handle[key]) for key in sorted(handle.files)]
    for i, trajectory in enumerate(trajectories):
        if trajectory.ndim != 2 or trajectory.shape[1] != 2:
            raise ValueError(f"mdshare trajectory {i} has unexpected shape {trajectory.shape}")
    return trajectories


trajectories = load_mdshare_dihedrals(DATA_DIR)
for i, trajectory in enumerate(trajectories):
    print(f"trajectory {i}: {trajectory.shape}, range=({trajectory.min():.3f}, {trajectory.max():.3f}) rad")

## 4. Reference distribution and metric helpers

In [ ]:
ANGLE_RANGE = [[-np.pi, np.pi], [-np.pi, np.pi]]


def histogram_probability(
    points: np.ndarray,
    weights: np.ndarray | None = None,
    bins: int = BINS,
    pseudocount: float = 1.0e-12,
) -> np.ndarray:
    hist, _, _ = np.histogram2d(
        points[:, 0],
        points[:, 1],
        bins=bins,
        range=ANGLE_RANGE,
        weights=weights,
    )
    hist = hist.astype(np.float64, copy=False) + pseudocount
    total = float(hist.sum())
    if total <= 0.0 or not np.isfinite(total):
        raise ValueError("Histogram total must be positive and finite")
    return hist / total


def kl_divergence(p_ref: np.ndarray, p_est: np.ndarray) -> float:
    p = np.asarray(p_ref, dtype=np.float64)
    q = np.asarray(p_est, dtype=np.float64)
    if p.shape != q.shape:
        raise ValueError("KL inputs must have the same shape")
    return float(np.sum(p * (np.log(p) - np.log(q))))


def coverage_fraction(points: np.ndarray, bins: int = BINS) -> float:
    hist, _, _ = np.histogram2d(points[:, 0], points[:, 1], bins=bins, range=ANGLE_RANGE)
    return float(np.count_nonzero(hist)) / float(hist.size)


reference_points = np.concatenate(trajectories, axis=0)
reference_probability = histogram_probability(reference_points)

fig, ax = plt.subplots(figsize=(5, 4))
image = ax.imshow(
    -np.log(reference_probability.T),
    origin="lower",
    extent=(-180, 180, -180, 180),
    aspect="equal",
)
ax.set_title("mdshare reference F(phi, psi), arbitrary units")
ax.set_xlabel("phi [deg]")
ax.set_ylabel("psi [deg]")
fig.colorbar(image, ax=ax, label="-log p")
fig.tight_layout()

## 5. VAMP-2 and retraining primitives

The notebook uses a weighted linear VAMP/TICA surrogate for the CV model. This is fast enough for the full factorial replay experiment and keeps the protocol falsifiable. In the real adaptive metadynamics run, replace `fit_weighted_vamp_model()` with the project's Deep-TICA training/export path and keep the same result schema.

In [ ]:
@dataclass(frozen=True)
class CVModel:
    mean: np.ndarray
    projection: np.ndarray
    train_vamp2: float
    validation_vamp2: float
    epochs_run: int
    train_seconds: float

    def transform(self, points: np.ndarray) -> np.ndarray:
        return (wrap_periodic(points) - self.mean) @ self.projection


def _normalise_weights(weights: np.ndarray | None, n: int) -> np.ndarray:
    if weights is None:
        return np.full(n, 1.0 / float(n), dtype=np.float64)
    raw = np.asarray(weights, dtype=np.float64).reshape(-1)
    if raw.shape[0] != n:
        raise ValueError("Weights must match the number of samples")
    if np.any(raw < 0.0) or not np.all(np.isfinite(raw)):
        raise ValueError("Weights must be finite and non-negative")
    total = float(raw.sum())
    if total <= 0.0:
        raise ValueError("Weights must sum to a positive value")
    return raw / total


def _inverse_sqrt(matrix: np.ndarray, reg: float = 1.0e-8) -> np.ndarray:
    values, vectors = np.linalg.eigh(matrix)
    values = np.maximum(values, reg)
    return (vectors / np.sqrt(values)) @ vectors.T


def vamp2_score_pairs(
    x0: np.ndarray,
    x1: np.ndarray,
    weights: np.ndarray | None = None,
    components: int = VAMP_COMPONENTS,
) -> float:
    if x0.shape != x1.shape:
        raise ValueError("VAMP pairs must have matching shapes")
    if x0.shape[0] < max(components + 2, 8):
        return float("nan")
    w = _normalise_weights(weights, x0.shape[0])
    mean0 = np.average(x0, axis=0, weights=w)
    mean1 = np.average(x1, axis=0, weights=w)
    c0 = x0 - mean0
    c1 = x1 - mean1
    c00 = (c0 * w[:, None]).T @ c0
    c11 = (c1 * w[:, None]).T @ c1
    c01 = (c0 * w[:, None]).T @ c1
    koopman = _inverse_sqrt(c00) @ c01 @ _inverse_sqrt(c11)
    singular_values = np.linalg.svd(koopman, compute_uv=False)
    return float(np.sum(singular_values[:components] ** 2))


def vamp2_score_window(points: np.ndarray, lag: int = LAG) -> float:
    if points.shape[0] <= lag + 8:
        return float("nan")
    x0 = points[:-lag]
    x1 = points[lag:]
    return vamp2_score_pairs(x0, x1)


def _fit_projection(x0: np.ndarray, x1: np.ndarray, weights: np.ndarray | None) -> tuple[np.ndarray, np.ndarray]:
    w = _normalise_weights(weights, x0.shape[0])
    mean0 = np.average(x0, axis=0, weights=w)
    mean1 = np.average(x1, axis=0, weights=w)
    c0 = x0 - mean0
    c1 = x1 - mean1
    c00 = (c0 * w[:, None]).T @ c0
    c11 = (c1 * w[:, None]).T @ c1
    c01 = (c0 * w[:, None]).T @ c1
    left = _inverse_sqrt(c00)
    right = _inverse_sqrt(c11)
    koopman = left @ c01 @ right
    u, _, _ = np.linalg.svd(koopman, full_matrices=False)
    projection = left @ u[:, :VAMP_COMPONENTS]
    return mean0, projection


def fit_weighted_vamp_model(
    points: np.ndarray,
    weights: np.ndarray | None,
    policy_name: str,
    lag: int = LAG,
) -> CVModel:
    params = TRAINING_POLICIES[policy_name]
    max_epochs = int(params["max_epochs"])
    early_stopping = bool(params["early_stopping"])

    if points.shape[0] <= lag + 32:
        raise ValueError("Training window is too short for the configured lag")

    x0_all = points[:-lag]
    x1_all = points[lag:]
    pair_weights = None if weights is None else np.asarray(weights[:-lag], dtype=np.float64)
    split = max(16, int(0.8 * x0_all.shape[0]))
    x0_train, x1_train = x0_all[:split], x1_all[:split]
    x0_val, x1_val = x0_all[split:], x1_all[split:]
    w_train = None if pair_weights is None else pair_weights[:split]
    w_val = None if pair_weights is None else pair_weights[split:]

    best_model: CVModel | None = None
    last_model: CVModel | None = None
    best_val = -np.inf
    stale_epochs = 0
    start = time.perf_counter()

    for epoch in range(1, max_epochs + 1):
        end = min(x0_train.shape[0], epoch * MAX_PAIRS_PER_EPOCH)
        if end < 16:
            continue
        train_slice = slice(0, end)
        w_epoch = None if w_train is None else w_train[train_slice]
        mean, projection = _fit_projection(x0_train[train_slice], x1_train[train_slice], w_epoch)
        train_cv0 = (x0_train[train_slice] - mean) @ projection
        train_cv1 = (x1_train[train_slice] - mean) @ projection
        val_cv0 = (x0_val - mean) @ projection
        val_cv1 = (x1_val - mean) @ projection
        train_score = vamp2_score_pairs(train_cv0, train_cv1, w_epoch)
        val_score = vamp2_score_pairs(val_cv0, val_cv1, w_val)
        elapsed = time.perf_counter() - start
        candidate = CVModel(mean, projection, train_score, val_score, epoch, elapsed)

        last_model = candidate
        if early_stopping:
            if val_score > best_val + EARLY_STOPPING_EPS:
                best_model = candidate
                best_val = val_score
                stale_epochs = 0
            else:
                stale_epochs += 1
        else:
            best_model = candidate

        if early_stopping and stale_epochs >= EARLY_STOPPING_PATIENCE:
            break

    if best_model is None:
        best_model = last_model
    if best_model is None:
        raise RuntimeError("VAMP model fitting did not produce a model")
    return best_model

## 6. Drift detector, bias ledger, and stream replay

`SimpleADWIN` is a compact ADWIN-style mean-shift detector for notebook use. It keeps an adaptive window and applies a Hoeffding-style split test. For production, replace it with the detector implementation selected for the package and keep the same trigger interface.

In [ ]:
class SimpleADWIN:
    def __init__(self, delta: float, min_window: int, max_window: int) -> None:
        self.delta = float(delta)
        self.min_window = int(min_window)
        self.max_window = int(max_window)
        self.window: deque[float] = deque()

    def update(self, value: float) -> bool:
        self.window.append(float(value))
        while len(self.window) > self.max_window:
            self.window.popleft()
        if len(self.window) < 2 * self.min_window:
            return False

        values = np.asarray(self.window, dtype=np.float64)
        n = values.size
        for cut in range(self.min_window, n - self.min_window + 1):
            left = values[:cut]
            right = values[cut:]
            eps = math.sqrt(0.5 * math.log(4.0 / self.delta) * (1.0 / left.size + 1.0 / right.size))
            if abs(float(left.mean() - right.mean())) > eps:
                for _ in range(cut):
                    self.window.popleft()
                return True
        return False


class GaussianBiasLedger:
    def __init__(self, sigma: float, height: float) -> None:
        self.sigma = float(sigma)
        self.height = float(height)
        self.centers: list[np.ndarray] = []

    def add_hill(self, center: np.ndarray) -> None:
        self.centers.append(wrap_periodic(np.asarray(center, dtype=np.float64)))

    def value(self, points: np.ndarray) -> np.ndarray:
        pts = wrap_periodic(np.atleast_2d(points))
        if not self.centers:
            return np.zeros(pts.shape[0], dtype=np.float64)
        centers = np.vstack(self.centers)
        diff = pts[:, None, :] - centers[None, :, :]
        diff = (diff + np.pi) % (2.0 * np.pi) - np.pi
        r2 = np.sum(diff * diff, axis=2)
        return self.height * np.exp(-0.5 * r2 / (self.sigma * self.sigma)).sum(axis=1)


def compute_bias_trace(points: np.ndarray) -> np.ndarray:
    ledger = GaussianBiasLedger(HILL_SIGMA_RAD, HILL_HEIGHT_KBT)
    bias_values = np.zeros(points.shape[0], dtype=np.float64)
    for frame_index, point in enumerate(points):
        bias_values[frame_index] = ledger.value(point)[0]
        if frame_index % HILL_STRIDE == 0:
            ledger.add_hill(point)
    return bias_values


def stable_reweighting_factors(bias_values: np.ndarray) -> np.ndarray:
    raw = BETA * np.asarray(bias_values, dtype=np.float64)
    shifted = raw - float(np.max(raw))
    weights = np.exp(shifted)
    if not np.all(np.isfinite(weights)) or float(weights.sum()) <= 0.0:
        raise ValueError("Invalid reweighting factors")
    return weights


@dataclass(frozen=True)
class Condition:
    trigger: str
    data_policy: str
    training_policy: str


def condition_seed(condition: Condition) -> int:
    text = f"{condition.trigger}|{condition.data_policy}|{condition.training_policy}"
    return sum((index + 1) * ord(char) for index, char in enumerate(text))


def select_training_data(
    stream: np.ndarray,
    bias_values: np.ndarray,
    end_frame: int,
    data_policy: str,
) -> tuple[np.ndarray, np.ndarray | None]:
    if data_policy in {"Full", "Reweighted-Full"}:
        start = 0
    elif data_policy in {"Window-W", "Reweighted-Window"}:
        start = max(0, end_frame - TRAIN_WINDOW)
    else:
        raise ValueError(f"Unknown data policy: {data_policy}")

    points = stream[start:end_frame]
    if data_policy.startswith("Reweighted"):
        weights = stable_reweighting_factors(bias_values[start:end_frame])
    else:
        weights = None
    return points, weights

## 7. Run the factorial replay experiment

In [ ]:
def estimate_replay_seconds_per_frame(sample: np.ndarray) -> float:
    start = time.perf_counter()
    _ = compute_bias_trace(sample[: min(sample.shape[0], 5_000)])
    elapsed = time.perf_counter() - start
    return elapsed / float(min(sample.shape[0], 5_000))


REPLAY_SECONDS_PER_FRAME = estimate_replay_seconds_per_frame(trajectories[0])
print(f"Replay calibration: {REPLAY_SECONDS_PER_FRAME:.3e} seconds/frame")


def draw_stream_segment(replica: int, rng: np.random.Generator) -> tuple[int, int, np.ndarray]:
    trajectory_index = replica % len(trajectories)
    trajectory = trajectories[trajectory_index]
    required = BUDGET_FRAMES + LAG + 1
    if trajectory.shape[0] < required:
        raise ValueError("Configured budget is longer than the available mdshare trajectory")
    start = int(rng.integers(0, trajectory.shape[0] - required + 1))
    return trajectory_index, start, trajectory[start : start + required]


def should_trigger(
    condition: Condition,
    frame_index: int,
    last_train_frame: int,
    current_score: float,
    score_at_last_train: float,
    detector: SimpleADWIN,
) -> bool:
    if condition.trigger == "Fixed-T":
        return frame_index - last_train_frame >= FIXED_TRIGGER_PERIOD
    if condition.trigger == "Threshold-delta":
        return score_at_last_train - current_score > THRESHOLD_DELTA
    if condition.trigger == "ADWIN":
        return detector.update(current_score)
    raise ValueError(f"Unknown trigger: {condition.trigger}")


def run_condition_replica(condition: Condition, replica: int) -> dict[str, float | int | str]:
    rng = np.random.default_rng(RNG_SEED + 10_000 * replica + condition_seed(condition))
    trajectory_index, start_frame, stream = draw_stream_segment(replica, rng)
    stream = stream[:BUDGET_FRAMES]
    bias_values = compute_bias_trace(stream)

    initial_frame = MONITOR_WINDOW
    initial_points, initial_weights = select_training_data(
        stream,
        bias_values,
        initial_frame,
        condition.data_policy,
    )
    model = fit_weighted_vamp_model(initial_points, initial_weights, condition.training_policy)
    score_at_last_train = vamp2_score_window(stream[initial_frame - MONITOR_WINDOW : initial_frame])
    last_train_frame = initial_frame
    retrain_count = 0
    total_train_seconds = model.train_seconds
    marginal_gain_sum = 0.0
    last_coverage = coverage_fraction(stream[:initial_frame])
    detector = SimpleADWIN(ADWIN_DELTA, ADWIN_MIN_WINDOW, ADWIN_MAX_WINDOW)
    score_trace: list[float] = []

    for frame_index in range(initial_frame + MONITOR_STRIDE, BUDGET_FRAMES + 1, MONITOR_STRIDE):
        window = stream[frame_index - MONITOR_WINDOW : frame_index]
        current_score = vamp2_score_window(window)
        if not np.isfinite(current_score):
            continue
        score_trace.append(current_score)
        if should_trigger(
            condition,
            frame_index,
            last_train_frame,
            current_score,
            score_at_last_train,
            detector,
        ):
            train_points, train_weights = select_training_data(
                stream,
                bias_values,
                frame_index,
                condition.data_policy,
            )
            model = fit_weighted_vamp_model(train_points, train_weights, condition.training_policy)
            retrain_count += 1
            total_train_seconds += model.train_seconds
            current_coverage = coverage_fraction(stream[:frame_index])
            train_equivalent_frames = max(model.train_seconds / REPLAY_SECONDS_PER_FRAME, 1.0)
            marginal_gain_sum += max(current_coverage - last_coverage, 0.0) / train_equivalent_frames
            last_coverage = current_coverage
            last_train_frame = frame_index
            score_at_last_train = current_score

    final_weights = None
    if condition.data_policy.startswith("Reweighted"):
        final_weights = stable_reweighting_factors(bias_values)
    estimated_probability = histogram_probability(stream, weights=final_weights)
    test_trajectory = trajectories[(trajectory_index + 1) % len(trajectories)]
    test_start = min(start_frame, test_trajectory.shape[0] - BUDGET_FRAMES - LAG - 1)
    test_points = test_trajectory[test_start : test_start + BUDGET_FRAMES]
    test_cv = model.transform(test_points)

    return {
        **asdict(condition),
        "replica": replica,
        "trajectory_index": trajectory_index,
        "start_frame": start_frame,
        "budget_frames": BUDGET_FRAMES,
        "kl_ref_to_est": kl_divergence(reference_probability, estimated_probability),
        "coverage": coverage_fraction(stream),
        "retrain_count": retrain_count,
        "total_train_seconds": total_train_seconds,
        "train_equivalent_frames": total_train_seconds / REPLAY_SECONDS_PER_FRAME,
        "marginal_gain_sum": marginal_gain_sum,
        "final_train_vamp2": model.train_vamp2,
        "final_validation_vamp2": model.validation_vamp2,
        "final_epochs_run": model.epochs_run,
        "test_vamp2": vamp2_score_window(test_cv),
        "monitor_vamp2_mean": float(np.mean(score_trace)) if score_trace else float("nan"),
        "monitor_vamp2_std": float(np.std(score_trace)) if score_trace else float("nan"),
    }


conditions = [
    Condition(trigger, data_policy, training_policy)
    for trigger, data_policy, training_policy in itertools.product(
        TRIGGER_POLICIES,
        DATA_POLICIES,
        TRAINING_POLICIES.keys(),
    )
]

results: list[dict[str, float | int | str]] = []
total_runs = len(conditions) * REPLICATES_PER_CONDITION
run_index = 0
for replica in range(REPLICATES_PER_CONDITION):
    for condition in conditions:
        run_index += 1
        print(f"[{run_index:03d}/{total_runs:03d}] {condition} replica={replica}")
        results.append(run_condition_replica(condition, replica))

results_df = pd.DataFrame(results)
results_path = OUTPUT_DIR / "adaptive_retraining_replay_results.csv"
results_df.to_csv(results_path, index=False)
print(f"Saved raw results to {results_path}")
results_df.head()

## 8. Aggregate results and choose strategies

Lower KL is better. Higher coverage, independent test VAMP-2, and marginal gain are better. A useful adaptive strategy should beat the fixed schedule at comparable total budget and should not buy small KL changes with excessive retraining overhead.

In [ ]:
metric_columns = [
    "kl_ref_to_est",
    "coverage",
    "retrain_count",
    "train_equivalent_frames",
    "marginal_gain_sum",
    "test_vamp2",
    "final_validation_vamp2",
    "final_epochs_run",
]

summary = (
    results_df.groupby(["trigger", "data_policy", "training_policy"], as_index=False)[metric_columns]
    .agg(["mean", "std"])
)
summary.columns = ["_".join(col).rstrip("_") for col in summary.columns.to_flat_index()]
summary = summary.sort_values(["kl_ref_to_est_mean", "coverage_mean"], ascending=[True, False])
summary_path = OUTPUT_DIR / "adaptive_retraining_replay_summary.csv"
summary.to_csv(summary_path, index=False)
print(f"Saved summary to {summary_path}")
summary.head(12)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

plot_df = summary.copy()
labels = plot_df["trigger"] + "\n" + plot_df["data_policy"] + "\n" + plot_df["training_policy"]
top = plot_df.head(12).copy()

axes[0].barh(range(top.shape[0]), top["kl_ref_to_est_mean"])
axes[0].set_yticks(range(top.shape[0]), top["trigger"] + " | " + top["data_policy"] + " | " + top["training_policy"])
axes[0].invert_yaxis()
axes[0].set_xlabel("KL(ref || estimate), mean")
axes[0].set_title("Best KL strategies")

axes[1].scatter(summary["train_equivalent_frames_mean"], summary["coverage_mean"], c=summary["kl_ref_to_est_mean"], cmap="viridis_r")
axes[1].set_xlabel("training cost [replay-equivalent frames]")
axes[1].set_ylabel("coverage")
axes[1].set_title("Coverage vs training overhead")

scatter = axes[2].scatter(summary["retrain_count_mean"], summary["test_vamp2_mean"], c=summary["marginal_gain_sum_mean"], cmap="plasma")
axes[2].set_xlabel("retraining events")
axes[2].set_ylabel("independent test VAMP-2")
axes[2].set_title("CV quality vs retraining count")
fig.colorbar(scatter, ax=axes[2], label="sum marginal gain")

fig.tight_layout()
plot_path = OUTPUT_DIR / "adaptive_retraining_replay_plots.png"
fig.savefig(plot_path, dpi=180)
print(f"Saved plot to {plot_path}")

## 9. Export protocol metadata

The JSON file records the fixed constants needed to compare this replay experiment with the later OpenMM/PLUMED adaptive metadynamics experiment. Replace `REPLAY_SECONDS_PER_FRAME` with a measured MD integration seconds-per-step value in the production run.

In [ ]:
protocol = {
    "source": "mdshare alanine-dipeptide-3x250ns-backbone-dihedrals.npz",
    "scope": "offline replay trigger/data/training-budget experiment; not biased MD",
    "budget_frames": BUDGET_FRAMES,
    "replicates_per_condition": REPLICATES_PER_CONDITION,
    "lag": LAG,
    "bins": BINS,
    "monitor_window": MONITOR_WINDOW,
    "monitor_stride": MONITOR_STRIDE,
    "train_window": TRAIN_WINDOW,
    "fixed_trigger_period": FIXED_TRIGGER_PERIOD,
    "threshold_delta": THRESHOLD_DELTA,
    "adwin_delta": ADWIN_DELTA,
    "hill_stride": HILL_STRIDE,
    "hill_height_kbt": HILL_HEIGHT_KBT,
    "hill_sigma_rad": HILL_SIGMA_RAD,
    "training_policies": TRAINING_POLICIES,
    "replay_seconds_per_frame": REPLAY_SECONDS_PER_FRAME,
    "outputs": {
        "raw_results": str(results_path),
        "summary": str(summary_path),
        "plot": str(plot_path),
    },
}

protocol_path = OUTPUT_DIR / "adaptive_retraining_replay_protocol.json"
protocol_path.write_text(json.dumps(protocol, indent=2), encoding="utf-8")
print(f"Saved protocol metadata to {protocol_path}")

## 10. How to move from this notebook to real adaptive metadynamics

1. Keep the grid: `Trigger x DataPolicy x TrainingPolicy x Replica`.
2. Replace `draw_stream_segment()` with an OpenMM/PLUMED loop that yields new `(phi, psi)` frames and the collection-time `V_bias(phi, psi, t)`.
3. Replace `compute_bias_trace()` with the actual PLUMED bias value recorded for each frame.
4. Replace `fit_weighted_vamp_model()` with the pmarlo Deep-TICA training and TorchScript export path.
5. Replace `REPLAY_SECONDS_PER_FRAME` with measured unbiased MD seconds per integration step, then keep reporting `train_equivalent_frames`.
6. Keep the mdshare reference distribution for `KL(p_ref || p_B)` and the held-out mdshare trajectory for independent VAMP-2.